# m20 lab · Geometry from catheter positions

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

A mapping system builds the chamber's shape from where the catheter has been. You sample an illustrative chamber near its wall, wrap the samples in a convex hull, reconstruct the outline by inflating, filling and shrinking, smooth a noisy surface without shrinking it, and measure volumes and surface distances.

| Part | What you do | Equation |
| --- | --- | --- |
| A | A chamber and catheter positions | setup |
| B | The convex hull | eq-m20-hull |
| C | Inflate, fill, shrink | eq-m20-closing |
| D | Smoothing without shrinking | eq-m20-smoothing |
| E | Volume and surface distance | eq-m20-volume, eq-m20-distance |

Lengths in mm. Parts A–C use a 2D slice on a 0.5 mm grid to keep the lab fast; the chamber (a disk with a notch and a stub) and the sampling are illustrative. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import geometry as G
from epml import meshes as MS

## Part A · A chamber and catheter positions

The chamber is a disk of radius 25 mm with a concave notch and a vein-like stub. The catheter mostly records positions
near the wall: sample random wall pixels and move them inward by up to 2 mm.

In [ ]:
ax, T = G.truth_mask(0.5, 50.0, 2)
rng = np.random.default_rng(20)
P = G.wall_samples(T, ax, 160, 2.0, rng)
plt.figure(figsize=(4.5, 4.5))
plt.contour(ax, ax, T.T.astype(float), [0.5], colors="gray")
plt.plot(P[:, 0], P[:, 1], ".k", ms=3)
plt.gca().set_aspect("equal"); plt.xlabel("x (mm)"); plt.ylabel("y (mm)"); plt.show()
print(f"true area {T.sum() * 0.25:.0f} mm^2, {len(P)} catheter positions")

In [ ]:
# Checks
from scipy import ndimage
depth = ndimage.distance_transform_edt(T) * 0.5
idx = np.clip(np.round((P - ax[0]) / 0.5).astype(int), 0, len(ax) - 1)
assert np.all(T[tuple(idx.T)]) and np.max(depth[tuple(idx.T)]) <= 3.0

## Part B · The convex hull

The convex hull is the smallest convex set containing the points (eq-m20-hull). It cannot follow a concavity: wall
points in the notch lie inside it, at a distance equal to the notch's depth, however many samples you collect.

In [ ]:
for n in (40, 160, 640):
    Pn = G.wall_samples(T, ax, n, 2.0, rng)
    H = G.hull_mask(Pn, ax)
    print(f"{n:4d} positions: hull area {H.sum() * 0.25:6.0f} mm^2, largest distance to the true wall {G.hausdorff(H, T, ax):.1f} mm")

In [ ]:
# Checks
assert G.hausdorff(G.hull_mask(G.wall_samples(T, ax, 640, 2.0, rng), ax), T, ax) > 6

## Part C · Inflate, fill, shrink

Inflate every position by a disk of radius $r$, fill the enclosed hole, then shrink by $r$ (a morphological closing with
hole filling, eq-m20-closing). If neighboring positions are farther apart than about $2r$ the inflated band has gaps and
nothing is enclosed; a large $r$ rounds off detail; inflating without shrinking moves the wall out by about $r$.

In [ ]:
print("positions   r    area    largest   mean   (inflate only: mean)")
results = {}
for n in (40, 80, 160, 320):
    Pn = G.wall_samples(T, ax, n, 2.0, np.random.default_rng(n))
    for r in (3.0, 5.0, 8.0):
        R = G.reconstruct(Pn, ax, r)
        D = G.reconstruct(Pn, ax, r, close=False)
        ok = R.sum() > 0.5 * T.sum()
        results[(n, r)] = (ok, G.mean_surface_distance(R, T, ax) if ok else np.inf)
        if ok:
            print(f"{n:6d}  {r:4.0f}  {R.sum() * 0.25:6.0f}   {G.hausdorff(R, T, ax):5.1f}   {results[(n, r)][1]:5.2f}   ({G.mean_surface_distance(D, T, ax):4.2f})")
        else:
            print(f"{n:6d}  {r:4.0f}   gaps: nothing enclosed")

In [ ]:
# Checks
assert not results[(40, 3.0)][0] and results[(320, 5.0)][0] and results[(320, 5.0)][1] < 1.0

## Part D · Smoothing without shrinking

Moving each vertex toward the mean of its neighbors by a fraction $\lambda$ (the umbrella operator) smooths noise but
shrinks the shape: a regular $n$-gon's radius falls by the factor $1 - \lambda a$, $a = 1 - \cos(2\pi/n)$, every step
(eq-m20-smoothing). Taubin's method alternates a shrinking step $\lambda$ and an inflating step $\mu < -\lambda$, giving
$(1 - \lambda a)(1 - \mu a) \approx 1$ for smooth shapes while still damping noise.

In [ ]:
V, nb = G.polygon(64, 20.0)
noisy = V * (1 + 0.07 * rng.uniform(-1, 1, (64, 1)))
for k in (10, 50, 200):
    L = G.laplacian(noisy, nb, 0.5, k)
    Tb = G.taubin(noisy, nb, 0.5, -0.53, k)
    print(f"{k:3d} steps: mean radius Laplacian {np.linalg.norm(L, axis=1).mean():6.2f}, Taubin {np.linalg.norm(Tb, axis=1).mean():6.2f} (from 20)")
Vs, Fs = MS.icosphere(level=3, radius=20.0)
nbs = G.mesh_neighbors(Fs, len(Vs))
Vn = Vs * (1 + 0.03 * rng.standard_normal((len(Vs), 1)))
v0 = G.mesh_volume(Vs, Fs)
print(f"sphere volume {v0:.0f}; after 30 Laplacian steps {G.mesh_volume(G.laplacian(Vn, nbs, 0.5, 30), Fs):.0f}; "
      f"after 30 Taubin steps {G.mesh_volume(G.taubin(Vn, nbs, 0.5, -0.53, 30), Fs):.0f}")

In [ ]:
# Checks
assert abs(np.linalg.norm(G.taubin(noisy, nb, 0.5, -0.53, 50), axis=1).mean() - 20) < 0.5
assert np.linalg.norm(G.laplacian(noisy, nb, 0.5, 200), axis=1).mean() < 19

## Part E · Volume and surface distance

For a closed, outward-oriented triangle mesh the divergence theorem gives the volume as $\frac16\sum
\mathbf v_a\cdot(\mathbf v_b\times\mathbf v_c)$ (eq-m20-volume). Compare two surfaces by the Hausdorff distance, the
worst case, and the mean surface distance (eq-m20-distance).

In [ ]:
for level in (1, 2, 3, 4):
    Vs, Fs = MS.icosphere(level=level, radius=10.0)
    print(f"icosphere level {level}: {len(Fs):5d} faces, volume {G.mesh_volume(Vs, Fs):8.2f} (sphere {4/3*np.pi*1000:.2f})")
t = np.linspace(0, 2 * np.pi, 720, endpoint=False)
A, B = 20 * np.c_[np.cos(t), np.sin(t)], 23 * np.c_[np.cos(t), np.sin(t)]
print("Hausdorff distance between circles of radius 20 and 23:", round(G.point_set_hausdorff(A, B), 3))

In [ ]:
# Checks
Vs, Fs = MS.icosphere(level=4, radius=10.0)
assert abs(G.mesh_volume(Vs, Fs) / (4/3*np.pi*1000) - 1) < 0.01

## Where to go next

- Repeat part C in 3D with many more positions and see how the number needed grows.
- Weight positions by contact (m13) or by respiratory phase (m19) before reconstructing.
- m21 registers this geometry to a CT or MRI segmentation; m24 colors it with activation times.